# 9.4 Specifying Data Using Pandas DataFrames

The dictionary-based methods of [Sections 9.2](9_2_data_in_dicts.ipynb) and [9.3](9_3_data_composite.ipynb) work well for hand-crafted data, but they become cumbersome once the data is large or comes from a structured source such as a spreadsheet. In those situations, a *Pandas DataFrame* is the more natural representation: a DataFrame is essentially a named, labeled table in which each column holds one parameter and the row index identifies the set element.

amplpy provides the `set_data()` method to load a DataFrame directly into AMPL. A single call to `set_data(df, "SET_NAME")` simultaneously defines the named AMPL set from the DataFrame index and assigns values to every parameter whose name matches a column in the DataFrame. This eliminates the need to write separate assignment statements for each parameter and makes it easy to keep the data in a tidy tabular form that is readable, editable, and readily sourced from CSV files or databases.

In [1]:
# install dependencies
%pip install -q amplpy pandas numpy polars

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Note: you may need to restart the kernel to use updated packages.
Licensed to AMPL Community Edition License for <uykun96@gmail.com>.


We use the same diet model as in previous sections.

In [2]:
%%writefile diet.mod

set NUTR;
set FOOD;
set LINKS within (NUTR cross FOOD);

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
param n_min {NUTR} >= 0;
param n_max {i in NUTR} >= n_min[i];
param amt {NUTR,FOOD} >= 0;

Overwriting diet.mod


In [3]:
ampl.read("diet.mod")

## 9.4.1 Building DataFrames for One-Dimensional Parameters

We construct three DataFrames. The first, `food_df`, contains one row per food item and three columns corresponding to the parameters `cost`, `f_min`, and `f_max`. The second, `nutr_df`, contains one row per nutrient and two columns for `n_min` and `n_max`. Both DataFrames use the set element as their row index, since `set_data()` reads the set membership from the index. The third DataFrame, `amt_df`, holds the two-dimensional nutrient-content matrix and will be handled separately in the next step.

In [7]:
import pandas as pd  # For handling data as DataFrames
import numpy as np   # For matrix operations  
    
# Create a DataFrame for food items
food_df = pd.DataFrame([           
    ("BEEF", 3.59, 2, 10),
    ("CHK",  2.59, 2, 10),
    ("FISH", 2.29, 2, 10),
    ("HAM",  2.89, 2, 10),
    ("MCH",  1.89, 2, 10),
    ("MTL",  1.99, 2, 10),
    ("SPG",  1.99, 2, 10),
    ("TUR",  2.49, 2, 10), ],
    columns=["FOOD", "cost", "f_min", "f_max"],).set_index("FOOD")

# DataFrame for nutrients
nutr_df = pd.DataFrame([ 
    ("A",   700,    20000),
    ("C",   700,    20000),
    ("B1",  700,    20000),
    ("B2",  700,    20000),
    ("NA",  0,      50000),
    ("CAL", 16000,  24000),],
    columns=["NUTR", "n_min", "n_max"],).set_index("NUTR")
    
# Nutrient content matrix
amt_df = pd.DataFrame(
    np.array([                          # np.array: The matrix is defined using NumPy's array function, which takes a nested list as input
    [60, 8, 8, 40, 15, 70, 25, 60],     # Nutrient 1 values for each food
    [20, 0, 10, 40, 35, 30, 50, 20],    # Nutrient 2 values for each food
    [10, 20, 15, 35, 15, 15, 25, 15],   # ...
    [15, 20, 10, 10, 15, 15, 15, 10],
    [928, 2180, 945, 278, 1182, 896, 1329, 1397],
    [295, 770, 440, 430, 315, 400, 379, 450],]),# Nutrient 6 values for each food
    columns=food_df.index.to_list(),    # The columns are labeled using the indices from food_df.
    index=nutr_df.index.to_list(),)     # The rows (index) are labeled using the indices from nutr_df

## 9.4.2 Loading Data into AMPL

With the DataFrames constructed, loading all of the one-dimensional data into AMPL is straightforward. The two `set_data()` calls handle `FOOD` and `NUTR` together with their associated parameters. For the two-dimensional parameter `amt`, we must also specify which `(nutrient, food)` pairs are valid. The model restricts these pairs to the set `LINKS`, which we derive from `amt_df` by listing all index combinations where the nutrient content is non-zero:

In [8]:
# Set data in AMPL
ampl.set_data(food_df, "FOOD")      # Definition of the "FOOD" set and parameters cost, f_min, f_max  
ampl.set_data(nutr_df, "NUTR")      # Definition of the "NUTR" set and parameters "n_min", "n_max"

# Define LINKS set
ampl.set['LINKS'] = [(nutrient, food)# Assigns a list of (nutrient, food) pairs to the LINKS
        for nutrient in amt_df.index  # Loops over the rows of amt_df. amt_df.index provides the row labels, which correspond to nutrients
        for food in amt_df.columns    #  Loops over the columns of amt_df. amt_df.columns provides the column labels, corresponding to food items.
        if amt_df.loc[nutrient, food] != 0] # Filters out combinations where the nutrient content for a specific food is 0

Finally, the `amt` values for those valid pairs are extracted and assigned:

In [9]:
# Filter and assign amt values
filtered_amt = {                                # Creates a dictionary to hold valid (nutrient, food) pairs with their corresponding values from amt_df
    (nutrient, food): amt_df.loc[nutrient, food]    # Fetches the value from amt_df for the (nutrient, food) pair
    for nutrient, food in ampl.set['LINKS'] }       # Loops over each (nutrient, food) pair in the 'LINKS' set
    
ampl.get_parameter("amt").set_values(filtered_amt) # Set the values for the parameter "amt" using "amt_

With these calls the diet model is fully populated. The key advantage of the Pandas approach is that the data is stored in a structured, tabular form that can be inspected, filtered, and transformed using the full power of Pandas before being passed to AMPL. When the data comes from a CSV file or a database query, `food_df` and `nutr_df` can be constructed directly from `pd.read_csv()` or `pd.read_sql()`, with no change to the AMPL-loading code.